# 10장 — 지형의 함정들: 안장점과 평원 🐎

말안장 지형에서 GD의 갇힘과 SGD의 탈출을 재현하고, 동전 논증을 셉니다.

**이 노트북의 목표**
1. 안장점의 정체: 방향별 곡률 부호가 섞인 임계점 (수치 미분으로 직접 확인)
2. 완벽한 GD는 능선 정중앙에 갇히고, SGD 노이즈는 탈출시킨다 (3장 순기능 재회)
3. 동전 논증: 방향 W개 전부 + 일 확률 (1/2)^W, 고차원 임계점은 거의 다 안장점

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch10')

## 1. 지형 정찰: 안장점의 방향별 곡률

무대: $L(a,b) = b^2/2 + (1-a^2)^2/4$. 안장점 $(0,0)$ 높이 0.25, 진짜 골짜기 $(\pm 1, 0)$ 높이 0.
안장점에서 방향별 곡률을 수치 미분으로 재 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

L = lambda a, b: b**2 / 2 + (1 - a**2)**2 / 4

# 등고선 지도 (신의 시점 — 1장의 그 특권)
A, B = np.meshgrid(np.linspace(-1.6, 1.6, 200), np.linspace(-3.3, 3.3, 200))
plt.figure(figsize=(6.5, 3.6))
plt.contourf(A, B, L(A, B), levels=30, cmap='Purples')
plt.colorbar(label='손실')
plt.plot(0, 0, 'wx', ms=10, mew=2); plt.text(0.06, 0.15, '안장점 (0.25)', color='w', fontsize=9)
plt.plot([1, -1], [0, 0], '*', color='gold', ms=13)
plt.title('말안장 지형: 세로로는 골짜기, 가로로는 능선'); plt.tight_layout(); plt.show()

# 안장점 (0,0)에서 방향별 곡률 (수치 2차 미분)
h = 1e-4
curv_a = (L(h, 0) - 2 * L(0, 0) + L(-h, 0)) / h**2   # 능선 방향
curv_b = (L(0, h) - 2 * L(0, 0) + L(0, -h)) / h**2   # 골짜기 방향
print(f'a(가로) 방향 곡률: {curv_a:+.3f}   b(세로) 방향 곡률: {curv_b:+.3f}')

assert curv_a < 0 < curv_b, '부호가 섞였다 — 기울기 0이지만 최소가 아닌 안장점'
print('\n✔ 확인: 방향별 곡률(실무 이름: 헤세 행렬의 고유값, 1부 7장) 부호가 섞임 = 안장점')

## 2. 실험 ① · 말안장 위의 여행자: GD vs SGD

위쪽 기슭 $(a,b)=(0,3)$에서 출발합니다. 능선 정중앙 코스입니다. 완벽한 나침반과 흔들리는 나침반의 운명은?

In [ ]:
def travel(sgd, seed=10, steps=150):
    rng = np.random.default_rng(seed)
    a, b = 0.0, 3.0
    losses, traj = [], [(a, b)]
    for t in range(steps):
        ga, gb = a**3 - a, b            # ∇L — 능선 정중앙(a=0)에선 ga가 정확히 0
        if sgd:
            ga += 0.08 * rng.standard_normal()
            gb += 0.08 * rng.standard_normal()
        a -= 0.15 * ga; b -= 0.15 * gb
        losses.append(L(a, b)); traj.append((a, b))
    return np.array(losses), np.array(traj)

gd_l, gd_t = travel(False)
sgd_l, sgd_t = travel(True)
escape = int(np.argmax(sgd_l < 0.1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].contourf(A, B, L(A, B), levels=30, cmap='Purples')
axes[0].plot(gd_t[:, 0], gd_t[:, 1], color='tab:red', lw=2, label='GD')
axes[0].plot(sgd_t[:, 0], sgd_t[:, 1], color='tab:green', lw=1.5, label='SGD')
axes[0].plot([1, -1], [0, 0], '*', color='gold', ms=12); axes[0].legend(); axes[0].set_title('자취')
axes[1].semilogy(gd_l, color='tab:red', label='GD')
axes[1].semilogy(sgd_l, color='tab:green', label='SGD')
axes[1].axhline(0.25, color='gray', ls=':', label='안장 높이 0.25')
axes[1].legend(); axes[1].set_title('손실 곡선: GD의 평원은 안장점 통과 중(9장 무늬)')
plt.tight_layout(); plt.show()

print(f'GD : 150스텝째 손실 {gd_l[-1]:.4f}, a = {gd_t[-1, 0]:.6f} (능선 정중앙 그대로)')
print(f'SGD: 탈출(loss<0.1) 스텝 {escape}, 최종 손실 {sgd_l[-1]:.4f}, a = {sgd_t[-1, 0]:.2f} (골짜기 도착)')

assert abs(gd_l[-1] - 0.25) < 1e-6 and abs(gd_t[-1, 0]) < 1e-12, 'GD: 옆으로 미는 성분이 정확히 0 — 영원히 갇힘'
assert sgd_l[-1] < 0.01 and escape < 100, 'SGD: 노이즈가 능선 밖으로 밀어내 탈출 (3장 순기능 재회)'
print('\n✔ 확인: 안장점은 갇힘이 아니라 지연의 함정. 노이즈가 공짜 탈출 티켓')

## 3. 실험 ② · 동전 세기: (1/2)^W

국소최소가 되려면 모든 방향의 곡률이 +여야 합니다. 방향별 부호를 반반 동전으로 치고 100개씩 뽑아 봅니다.

In [ ]:
def draw_criticals(W, n=100, seed=7):
    rng = np.random.default_rng(seed)
    return sum(1 for _ in range(n) if (rng.random(W) < 0.5).all())

print(' W    이론 (1/2)^W        100개 중 국소최소')
counts = {}
for W in [1, 2, 4, 10, 20]:
    counts[W] = draw_criticals(W)
    print(f'{W:3d}   {0.5**W:.7f}   {counts[W]:3d}개')

assert 15 <= counts[2] <= 35, 'W=2: 약 4분의 1이 국소최소'
assert counts[10] <= 2 and counts[20] == 0, 'W가 조금만 커져도 국소최소는 씨가 마른다'
assert 0.5**10 == 1 / 1024, '(1/2)^10 = 1/1024'
print(f'\nW = 30이면 (1/2)^30 ≈ 10억분의 1, LLM(수십억 차원)이면 사실상 0')
print('✔ 확인: 고차원 임계점은 거의 전부 안장점. 저차원의 공포(웅덩이)가 뒤집힌다 (7부 4장 계열)')

## 4. 관찰 정리 + 🤖 AI 연결

- **안장점** = 기울기 0인데 방향별 곡률 부호(고유값, 1부 7장)가 섞인 임계점. 발밑만 보는
  여행자에겐 바닥과 구분 불가. 갇힘(국소최소)이 아니라 **지연**의 함정: 탈출구(− 곡률 방향)가 있다.
- **GD vs SGD**: 능선 정중앙의 GD는 옆으로 미는 성분이 정확히 0이라 영원히 갇히고, SGD의
  노이즈는 대칭을 깨 탈출시킨다(3장의 순기능). GD 곡선의 평평 구간 = 9장 평원 무늬의 원인 하나.
- **동전 논증**: P(모든 방향 +) = (1/2)^W. 수십억 차원이면 사실상 0, 임계점은 거의 전부 안장점.
  (정직한 주석: 실제 곡률은 공평한 동전이 아니고, 손실 낮은 임계점일수록 + 비율↑.
  깊은 곳 임계점은 거의 진짜 최소라는 더 좋은 소식.)
- **그림 완성**: 수십억 차원 훈련이 SGD로 되는 이유. 웅덩이는 드물고, 안장은 노이즈·모멘텀이
  밀어 주고, 나머지는 다이얼(2~6장)이 해결.

**다음 챕터**: 11장(🎯 캡스톤). 진짜 파인튜닝 설정 파일을 처음부터 끝까지 읽어 내는 졸업 시험.